In [1]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
#import dtale

from scipy.stats import chi2_contingency


In [2]:
# Rutas
RUTA_LECTURA = "/Users/minu/python/Analisis_Banca/Data/06-08-2026/06-08-2026_Clean.csv"
#RUTA_ESCRITURA = 

In [3]:
# Cargar datos 
df_marketing = pd.read_csv(RUTA_LECTURA) #, sep=";"

In [37]:
display(df_marketing.dtypes)

id                       int64
age                      int64
job                     object
marital                 object
education               object
default                  int64
balance                  int64
housing                  int64
loan                     int64
contact                 object
day                      int64
month                   object
duration                 int64
campaign                 int64
pdays                    int64
previous                 int64
poutcome                object
deposit                  int64
debt_profile            object
balance_tier            object
job_group               object
month_num                int64
year                     int64
date            datetime64[ns]
day_of_week           category
dtype: object

In [4]:
# --- CREACIÓN DE VARIABLES NECESARIAS 

# Perfil de deuda 
df_marketing['debt_profile'] = np.where(
    (df_marketing['housing'] == 1) | (df_marketing['loan'] == 1), 'With Debt', 'No Debt')

# Tramos de Balance 
condiciones_balance = [
    (df_marketing['balance'] <= 0),
    (df_marketing['balance'] > 0) & (df_marketing['balance'] <= 556),
    (df_marketing['balance'] > 556) & (df_marketing['balance'] <= 2000),
    (df_marketing['balance'] > 2000)
]
opciones_balance = ['Negative or Zero', 'Low Balance', 'Medium Balance', 'High Balance']

df_marketing['balance_tier'] = np.select(condiciones_balance, opciones_balance, default='Negative or Zero')

# Mapeo de Job a 5 Macro-categorías (aquí separé en un principio a unemployed a una categoría aparte. 
# Pero me dio como resultado que no era distinto a retired y student, entonces van en la misma categoria)
def agrupar_job(job):
    job = str(job).lower().strip()
    if job in ['admin.', 'management','blue-collar' ]:
        return 'Asalariados'
    elif job in ['technician', 'services', 'housemaid']:
        return 'Operativos'
    elif job in ['self-employed', 'entrepreneur']:
        return 'Independientes'
    elif job in ['retired', 'student', 'unemployed']:
        return 'Inactivos'
    else:
        return 'Unknown'
df_marketing['job_group'] = df_marketing['job'].apply(agrupar_job)


In [5]:
# Convertir meses de texto a números
meses = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}

df_marketing["month_num"] = df_marketing["month"].map(meses)

# Calcular los años basándose en el cambio de diciembre a enero
years = []
current_year = 2008
previous_month = df_marketing["month_num"].iloc[0]

for month in df_marketing["month_num"]:
    if previous_month == 12 and month == 1:
        current_year += 1

    years.append(current_year)
    previous_month = month

df_marketing["year"] = years

# Crear columna de fecha completa
df_marketing["date"] = pd.to_datetime({"year": df_marketing["year"], "month": df_marketing["month_num"], "day": df_marketing["day"]}, errors="coerce")

# Obtener y traducir el día de la semana en orden cronológico
dias_traducidos = {"Monday": "Lunes", "Tuesday": "Martes", "Wednesday": "Miércoles", "Thursday": "Jueves", "Friday": "Viernes", "Saturday": "Sábado", "Sunday": "Domingo"}
orden = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
df_marketing["day_of_week"] = pd.Categorical(df_marketing["date"].dt.day_name().map(dias_traducidos), categories=orden, ordered=True)

# Imprimir el reporte de control de fechas
print(f"Fechas nulas: {df_marketing['date'].isna().sum()}")
print(f"Rango: {df_marketing['date'].min()} a {df_marketing['date'].max()}")


Fechas nulas: 0
Rango: 2008-05-05 00:00:00 a 2010-12-29 00:00:00


In [6]:
# Conteo de total de dias de la semana del dataset
df_marketing['day_of_week'].value_counts()

Jueves       2390
Viernes      2259
Miércoles    2126
Martes       1410
Sábado       1303
Lunes         861
Domingo       813
Name: day_of_week, dtype: int64

In [25]:
df_marketing['day_of_week'].value_counts().sum()

11162

In [ ]:
# dtale.show(df_marketing)


In [7]:
# Diagnóstico: composición de job_group por día para saber porque Lunes tiene tan buen comportamiento
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["job_group"],
    normalize="index"
) * 100

print(composicion.round(1))

job_group    Asalariados  Inactivos  Independientes  Operativos  Unknown
day_of_week                                                             
Lunes               47.7       19.3             6.7        25.6      0.7
Martes              50.9       16.9             6.0        25.7      0.6
Miércoles           52.0       14.9             6.3        26.2      0.7
Jueves              51.1       13.3             6.9        27.9      0.8
Viernes             54.4       12.8             5.4        26.9      0.5
Sábado              55.2        9.2             7.0        28.4      0.2
Domingo             54.4        5.9             9.7        29.3      0.7


In [8]:
# Diagnóstico: composición de debt_profile por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["debt_profile"],
    normalize="index"
) * 100

print(composicion.round(1))

debt_profile  No Debt  With Debt
day_of_week                     
Lunes            61.0       39.0
Martes           51.1       48.9
Miércoles        48.4       51.6
Jueves           47.6       52.4
Viernes          45.9       54.1
Sábado           37.5       62.5
Domingo          39.1       60.9


In [9]:
# Diagnóstico: composición de poutcome por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["poutcome"],
    normalize="index"
) * 100

print(composicion.round(1))

poutcome     failure  no_campaign  other  success
day_of_week                                      
Lunes           12.9         62.5    6.7     17.9
Martes          17.3         60.7    6.8     15.2
Miércoles       10.0         76.1    3.4     10.5
Jueves          11.3         72.9    5.2     10.6
Viernes         10.5         74.9    5.6      8.9
Sábado           9.7         84.7    4.0      1.7
Domingo          3.3         95.4    1.1      0.1


In [10]:
# Diagnóstico: composición de balance_tier por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["balance_tier"],
    normalize="index"
) * 100

print(composicion.round(1))

balance_tier  High Balance  Low Balance  Medium Balance  Negative or Zero
day_of_week                                                              
Lunes                 25.4         33.8            31.6               9.2
Martes                24.3         35.5            29.9              10.4
Miércoles             21.0         36.9            27.2              14.8
Jueves                22.2         36.8            28.8              12.2
Viernes               21.9         38.2            27.0              12.9
Sábado                19.0         38.5            26.6              15.9
Domingo               19.8         40.0            24.0              16.2


In [11]:
# Diagnóstico: composición de contact por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["contact"],
    normalize="index"
) * 100

print(composicion.round(1))

contact      cellular  telephone  unknown
day_of_week                              
Lunes            82.3        8.2      9.4
Martes           86.2        8.7      5.1
Miércoles        71.6        7.1     21.3
Jueves           74.1        6.7     19.2
Viernes          72.1        6.4     21.5
Sábado           65.1        7.8     27.1
Domingo          42.8        2.8     54.4


In [10]:
# Forzar el orden de los días de la semana
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
df_marketing["day_of_week"] = pd.Categorical(df_marketing["day_of_week"], categories=dias, ordered=True)

# Calcular e imprimir la tabla de días calendario por año
tabla_dias = pd.crosstab(index=df_marketing['year'], columns=df_marketing['day_of_week'])
print("\n días calendario por año \n")
print(tabla_dias)

# Calcular e imprimir la tabla de llamadas totales por año
tabla_llamadas = df_marketing.groupby(['year', 'day_of_week'], observed=False)['campaign'].sum().unstack(fill_value=0)
print("\n \n llamadas totales por año \n")
print(tabla_llamadas)



 días calendario por año 

day_of_week  Lunes  Martes  Miércoles  Jueves  Viernes  Sábado  Domingo
year                                                                   
2008           187     184        248     239      227       1        3
2009           393     373        372     460      368       0        0
2010           281     853       1506    1691     1664    1302      810

 
 llamadas totales por año 

day_of_week  Lunes  Martes  Miércoles  Jueves  Viernes  Sábado  Domingo
year                                                                   
2008           607     492        727     672      641       1        3
2009           802     629        661     824      726       0        0
2010           568    1774       4093    4016     4241    4000     2522


In [11]:
# CHI-CUADRADO DE INDEPENDENCIA

# Crear tabla de cruce (usando la columna mapeada 'deposit')
tabla = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["deposit"].map({1: "Convierte", 0: "No convierte"})
)
print("TABLA CONTINGENCIA:\n", tabla)

# Prueba estadística Chi-cuadrado y validación de resultados 
chi2, p, _, esperadas = chi2_contingency(tabla)
print(f"\nCHI2: {chi2:.4f} | P-valor: {p:.4f}")
print("Conclusión:", "Significativo" if p < 0.05 else "No asociado")
print(f"Celdas esperadas < 5: {(esperadas < 5).sum()}")

# Tasa de conversión porcentual por día 
tasa = df_marketing.groupby("day_of_week", observed=True)["deposit"].agg(conv="sum", total="count")
tasa["tasa_%"] = (tasa["conv"] / tasa["total"] * 100).round(2)
print("\nTASA CONVERSIÓN:\n", tasa)

# Residuos estandarizados (para identificar qué días contribuyen más a la asociación)
residuos = pd.DataFrame(
    (tabla.values - esperadas) / np.sqrt(esperadas),
    index=tabla.index,
    columns=tabla.columns
)
print("\nRESIDUOS ESTANDARIZADOS:\n", residuos.round(3))

TABLA CONTINGENCIA:
 deposit      Convierte  No convierte
day_of_week                         
Lunes              812            49
Martes             949           461
Miércoles         1071          1055
Jueves            1205          1185
Viernes           1010          1249
Sábado             164          1139
Domingo             78           735

CHI2: 2106.3355 | P-valor: 0.0000
Conclusión: Significativo
Celdas esperadas < 5: 0

TASA CONVERSIÓN:
              conv  total  tasa_%
day_of_week                     
Lunes         812    861   94.31
Martes        949   1410   67.30
Miércoles    1071   2126   50.38
Jueves       1205   2390   50.42
Viernes      1010   2259   44.71
Sábado        164   1303   12.59
Domingo        78    813    9.59

RESIDUOS ESTANDARIZADOS:
 deposit      Convierte  No convierte
day_of_week                         
Lunes           20.003       -18.982
Martes          10.867       -10.312
Miércoles        2.004        -1.902
Jueves           2.155        -2.

In [12]:
# ---MODELO REGRESIÓN LOGÍSTICA---

# Establecemos 'Domingo', 'inactivos', 'With Debt' y 'Negative or Zero' 
# como las categorías base de comparación neutral para cada variable categórica respectivamente

formula_timing = """
deposit ~ C(day_of_week, Treatment(reference='Domingo')) 
         + C(job_group, Treatment(reference='Inactivos')) 
         + C(debt_profile, Treatment(reference='With Debt')) 
         + C(balance_tier, Treatment(reference='Negative or Zero')) 
         + C(poutcome)
         + C(contact, Treatment(reference="telephone"))
"""

modelo_timing = smf.logit(formula_timing, data=df_marketing).fit()

# Extracción y Estructuración de Resultados (Odds Ratios)
resultados_completos = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_timing.params),
    'P-valor': modelo_timing.pvalues.round(4)
})

Optimization terminated successfully.
         Current function value: 0.522112
         Iterations 7


In [13]:
# Ver el resumen estadístico completo
modelo_timing = smf.logit(formula_timing, data=df_marketing).fit()
print(modelo_timing.summary())

Optimization terminated successfully.
         Current function value: 0.522112
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                11162
Model:                          Logit   Df Residuals:                    11142
Method:                           MLE   Df Model:                           19
Date:                Fri, 12 Jun 2026   Pseudo R-squ.:                  0.2453
Time:                        12:03:18   Log-Likelihood:                -5827.8
converged:                       True   LL-Null:                       -7721.6
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                                 coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------

In [47]:
# Resultados (Odds Ratios)
display(resultados_completos)

,Odds Ratio (OR),P-valor
Intercept,0.093461,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Lunes]",98.475556,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Martes]",11.257261,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Miércoles]",6.581487,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Jueves]",6.420335,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Viernes]",5.193277,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Sábado]",1.059198,0.7003
"C(job_group, Treatment(reference='Inactivos'))[T.Asalariados]",0.661022,0.0000
"C(job_group, Treatment(reference='Inactivos'))[T.Independientes]",0.597647,0.0000
"C(job_group, Treatment(reference='Inactivos'))[T.Operativos]",0.602308,0.0000


___

# VISUALIZACIONES

In [14]:
# Resultados renombrados
nombres_legibles = {
    "Intercept": "Intercept",
    "C(day_of_week, Treatment(reference='Domingo'))[T.Lunes]":     "Lunes",
    "C(day_of_week, Treatment(reference='Domingo'))[T.Martes]":    "Martes",
    "C(day_of_week, Treatment(reference='Domingo'))[T.Miércoles]": "Miércoles",
    "C(day_of_week, Treatment(reference='Domingo'))[T.Jueves]":    "Jueves",
    "C(day_of_week, Treatment(reference='Domingo'))[T.Viernes]":   "Viernes",
    "C(day_of_week, Treatment(reference='Domingo'))[T.Sábado]":    "Sábado",
    "C(job_group, Treatment(reference='Inactivos'))[T.Asalariados]":   "Asalariados",
    "C(job_group, Treatment(reference='Inactivos'))[T.Independientes]":"Independientes",
    "C(job_group, Treatment(reference='Inactivos'))[T.Operativos]":    "Operativos",
    "C(job_group, Treatment(reference='Inactivos'))[T.Unknown]":       "Desconocido (job)",
    "C(debt_profile, Treatment(reference='With Debt'))[T.No Debt]":    "Sin Deuda",
    "C(balance_tier, Treatment(reference='Negative or Zero'))[T.High Balance]":   "Balance Alto",
    "C(balance_tier, Treatment(reference='Negative or Zero'))[T.Low Balance]":    "Balance Bajo",
    "C(balance_tier, Treatment(reference='Negative or Zero'))[T.Medium Balance]": "Balance Medio",
    "C(poutcome)[T.no_campaign]": "Sin Campaña Previa",
    "C(poutcome)[T.other]":       "Otro Resultado Previo",
    "C(poutcome)[T.success]":     "Éxito Campaña Previa",
    'C(contact, Treatment(reference="telephone"))[T.cellular]': "Contacto Celular",
    'C(contact, Treatment(reference="telephone"))[T.unknown]':  "Contacto Desconocido",
}

# Crear df limpio con solo Odds Ratio
df_or = (
    resultados_completos[["Odds Ratio (OR)"]]
    .rename(index=nombres_legibles)
)

display(df_or)

,Odds Ratio (OR)
Intercept,0.093461
Lunes,98.475556
Martes,11.257261
Miércoles,6.581487
Jueves,6.420335
Viernes,5.193277
Sábado,1.059198
Asalariados,0.661022
Independientes,0.597647
Operativos,0.602308


In [15]:
# Separar en DataFrames por grupo de variables

df_dia_semana = df_or.loc[["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]]

df_job_group = df_or.loc[["Asalariados", "Independientes", "Operativos", "Desconocido (job)"]]

df_balance = df_or.loc[["Sin Deuda", "Balance Alto", "Balance Bajo", "Balance Medio"]]

df_outcome = df_or.loc[["Sin Campaña Previa", "Otro Resultado Previo", "Éxito Campaña Previa"]]

df_contacto = df_or.loc[["Contacto Celular", "Contacto Desconocido"]]


In [16]:
# Fila de referencia
ref = pd.DataFrame({"Odds Ratio (OR)": [1.0]})

# Día de la semana — referencia: Domingo
ref_domingo = ref.copy()
ref_domingo.index = ["Domingo (ref.)"]
df_dia_semana = pd.concat([ref_domingo, df_dia_semana])

# Grupo de trabajo — referencia: Inactivos
ref_inactivos = ref.copy()
ref_inactivos.index = ["Inactivos (ref.)"]
df_job_group = pd.concat([ref_inactivos, df_job_group])

# Balance — referencia: Negativo o Cero
ref_balance = ref.copy()
ref_balance.index = ["Negativo o Cero (ref.)"]
df_balance = pd.concat([ref_balance, df_balance])

# Contacto — referencia: Telephone
ref_telefono = ref.copy()
ref_telefono.index = ["Teléfono (ref.)"]
df_contacto = pd.concat([ref_telefono, df_contacto])

display(df_dia_semana)
display(df_job_group)
display(df_balance)
display(df_outcome)
display(df_contacto)

,Odds Ratio (OR)
Domingo (ref.),1.000000
Lunes,98.475556
Martes,11.257261
Miércoles,6.581487
Jueves,6.420335
Viernes,5.193277
Sábado,1.059198


,Odds Ratio (OR)
Inactivos (ref.),1.000000
Asalariados,0.661022
Independientes,0.597647
Operativos,0.602308
Desconocido (job),0.412913


,Odds Ratio (OR)
Negativo o Cero (ref.),1.000000
Sin Deuda,1.818678
Balance Alto,1.882176
Balance Bajo,1.178097
Balance Medio,1.450466


,Odds Ratio (OR)
Sin Campaña Previa,0.967401
Otro Resultado Previo,1.289889
Éxito Campaña Previa,7.836308


,Odds Ratio (OR)
Teléfono (ref.),1.000000
Contacto Celular,1.416917
Contacto Desconocido,0.675475


In [17]:
# Cruce day_of_week x job_group
tasa_cruce = (
    df_marketing
    .groupby(["day_of_week", "job_group"], observed=True)["deposit"]
    .agg(conv="sum", total="count")
    .reset_index()
)
tasa_cruce["tasa_%"] = (tasa_cruce["conv"] / tasa_cruce["total"] * 100).round(2)


display(tasa_cruce.head())


,day_of_week,job_group,conv,total,tasa_%
0,Lunes,Asalariados,391,411,95.13
1,Lunes,Inactivos,148,166,89.16
2,Lunes,Independientes,56,58,96.55
3,Lunes,Operativos,214,220,97.27
4,Lunes,Unknown,3,6,50.00


In [18]:
# imports gráficos
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go

# DarkMode Plotly
pio.templates.default = "plotly_dark"
from plotly.subplots import make_subplots # permite visualizar varios gráficos

pio.templates["custom"] = pio.templates["plotly_dark"]
pio.templates["custom"].layout.paper_bgcolor = "#050a30"
pio.templates["custom"].layout.plot_bgcolor  = "#050a30"
pio.templates.default = "custom"

In [19]:
# Tasa de conversión porcentual por día 
tasa = df_marketing.groupby("day_of_week", observed=True)["deposit"].agg(conv="sum", total="count")
tasa["tasa_%"] = (tasa["conv"] / tasa["total"] * 100).round(2)


tasa_ordenada = (
    tasa["tasa_%"]
    .reindex(dias)
    .tolist()
)

print("\nTASA CONVERSIÓN:\n", tasa)
display(tasa_ordenada)


TASA CONVERSIÓN:
              conv  total  tasa_%
day_of_week                     
Lunes         812    861   94.31
Martes        949   1410   67.30
Miércoles    1071   2126   50.38
Jueves       1205   2390   50.42
Viernes      1010   2259   44.71
Sábado        164   1303   12.59
Domingo        78    813    9.59


[94.31, 67.3, 50.38, 50.42, 44.71, 12.59, 9.59]

In [20]:
# gráfico "Tasa de conversión (%) x dia ""

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Barras: tasa de conversión
fig.add_trace(
    go.Bar(
        x=dias,
        y=tasa_ordenada,
        name="Tasa de conversión (%)",
        marker_color="teal",
        opacity=0.95,
        text=tasa_ordenada,
    ),
    secondary_y=False,
)


fig.update_layout(
    title="Tasa de conversión por día de la semana",

)
fig.update_yaxes(
    title_text="Tasa de conversión (%)",
    range=[0, 110],
    secondary_y=False,
)

fig.show()

In [21]:
# Tasa de Conversión por Día de la Semana y Grupo de Trabajo
fig = px.bar(
    tasa_cruce,
    x="day_of_week",
    y="tasa_%",
    color="job_group",
    barmode="group",
    title="Tasa de Conversión por Día de la Semana y Grupo de Trabajo",
    labels={
        "day_of_week": "Día de la Semana",
        "tasa_%":      "Tasa de Conversión (%)",
        "job_group":   "Grupo de Trabajo"
    },
    category_orders={"day_of_week": dias},
    hover_data={"conv": True, "total": True}
)

fig.update_layout(
    xaxis_title="Día de la Semana",
    yaxis_title="Tasa de Conversión (%)",
    yaxis=dict(range=[0, 105]),
    legend_title_text="Grupo de Trabajo",
    bargap=0.2,
    bargroupgap=0.05
)

fig.show()

In [22]:
# Preparar OR por día

df_dia_semana_reset = df_dia_semana.reset_index()
df_dia_semana_reset.columns = ["day_of_week", "OR"]
df_dia_semana_reset["day_of_week"] = df_dia_semana_reset["day_of_week"].str.replace(" (ref.)", "", regex=False)

# Ordenar por día
df_dia_semana_reset["day_of_week"] = pd.Categorical(
    df_dia_semana_reset["day_of_week"], categories=dias, ordered=True
)
df_dia_semana_reset = df_dia_semana_reset.sort_values("day_of_week")


In [23]:
# Tasa de Conversión y Odds Ratio por Día de la Semana

# Figura con doble eje Y
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Barras por grupo de trabajo
for grupo in tasa_cruce["job_group"].unique():
    df_grupo = tasa_cruce[tasa_cruce["job_group"] == grupo]
    fig.add_trace(
        go.Bar(
            x=df_grupo["day_of_week"],
            y=df_grupo["tasa_%"],
            name=grupo,
            hovertemplate="Día: %{x}<br>Tasa: %{y}%<extra>" + grupo + "</extra>"
        ),
        secondary_y=False
    )

# Línea OR
fig.add_trace(
    go.Scatter(
        x=df_dia_semana_reset["day_of_week"],
        y=df_dia_semana_reset["OR"],
        name="Odds Ratio",
        mode="lines+markers",
        line=dict(color="white", width=2, dash="dash"),
        marker=dict(size=8)
    ),
    secondary_y=True
)

# Anotación referencia Domingo
fig.add_annotation(
    x="Domingo",
    y=1.0,
    text="Ref.<br>(Domingo)", # = 1.0
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
    font=dict(size=11, color="white"),
    secondary_y=True
)

fig.update_layout(
    title="Tasa de Conversión y Odds Ratio por Día de la Semana",
    barmode="group",
    bargap=0.2,
    bargroupgap=0.05,
    xaxis=dict(categoryorder="array", categoryarray=dias),
    #legend_title_text="Grupo"
)

fig.update_yaxes(title_text="Tasa de Conversión (%)", range=[0, 110], secondary_y=False)
fig.update_yaxes(title_text="Odds Ratio", range=[0, 110], secondary_y=True)

fig.show()

In [24]:
# tabla cruce: dia de la semana, tipo de contacto y trabajo
tasa_cruce_canal = (
    df_marketing
    .groupby(["day_of_week", "contact", "job_group"], observed=True)["deposit"]
    .agg(conv="sum", total="count")
    .reset_index()
)
tasa_cruce_canal["tasa_%"] = (tasa_cruce_canal["conv"] / tasa_cruce_canal["total"] * 100).round(2)

print(tasa_cruce_canal.head(20))

   day_of_week    contact       job_group  conv  total  tasa_%
0        Lunes   cellular     Asalariados   321    338   94.97
1        Lunes   cellular       Inactivos   121    134   90.30
2        Lunes   cellular  Independientes    44     46   95.65
3        Lunes   cellular      Operativos   181    186   97.31
4        Lunes   cellular         Unknown     3      5   60.00
5        Lunes  telephone     Asalariados    23     24   95.83
6        Lunes  telephone       Inactivos    24     27   88.89
7        Lunes  telephone  Independientes     7      7  100.00
8        Lunes  telephone      Operativos    12     12  100.00
9        Lunes  telephone         Unknown     0      1    0.00
10       Lunes    unknown     Asalariados    47     49   95.92
11       Lunes    unknown       Inactivos     3      5   60.00
12       Lunes    unknown  Independientes     5      5  100.00
13       Lunes    unknown      Operativos    21     22   95.45
14      Martes   cellular     Asalariados   389    620 

In [25]:
# Ordenar días
tasa_cruce_canal["day_of_week"] = pd.Categorical(
    tasa_cruce_canal["day_of_week"], categories=dias, ordered=True
)
tasa_cruce_canal = tasa_cruce_canal.sort_values(["day_of_week", "contact", "job_group"])

# Faceted bar chart: panel por canal, apilado por grupo de trabajo
fig = px.bar(
    tasa_cruce_canal,
    x="day_of_week",
    y="tasa_%",
    color="job_group",
    color_discrete_sequence=px.colors.qualitative.Set3,
    barmode="stack",
    facet_col="contact",
    title="Tasa de Conversión por Día, Canal y Grupo de Trabajo",
    text="tasa_%",
    labels={
        "day_of_week": "Día de la Semana",
        "tasa_%":      "Tasa de Conversión (%)",
        "job_group":   " ",
        "contact":     "Canal",
    },
    category_orders={
        "day_of_week": dias,
        "contact":     ["cellular", "telephone", "unknown"]
    },
    hover_data={"conv": True, "total": True}
)

fig.update_layout(
   legend_title_text="",
    bargap=0.2,
)

# Limpiar títulos de facetas


fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].capitalize()))

fig.update_yaxes(range=[0, 410], secondary_y=False)
fig.update_traces(textfont_size=10)

fig.show()

In [27]:
#heatmap
# Ordenar días
tasa_cruce_canal["day_of_week"] = pd.Categorical(
    tasa_cruce_canal["day_of_week"], categories=dias, ordered=True
)

# Un heatmap por canal
canales = tasa_cruce_canal["contact"].unique()

fig = make_subplots(
    rows=1, cols=len(canales),
    subplot_titles=[c.capitalize() for c in ["cellular", "telephone", "unknown"]]
)

for i, canal in enumerate(["cellular", "telephone", "unknown"], start=1):
    df_canal = (
        tasa_cruce_canal[tasa_cruce_canal["contact"] == canal]
        .pivot(index="job_group", columns="day_of_week", values="tasa_%")
    )

    fig.add_trace(
        go.Heatmap(
            z=df_canal.values,
            x=df_canal.columns.tolist(),
            y=df_canal.index.tolist(),
            colorscale="viridis",
            zmin=0, zmax=100,
            text=df_canal.values.round(0),
            texttemplate="%{text}%",
            showscale=False,
    
            colorbar=dict(title="Tasa (%)") if i == 3 else None
        ),
        row=1, col=i
    )

fig.update_layout(title="Heatmap: Tasa de Conversión por Día, Grupo de Trabajo y Canal")

fig.show()

In [28]:
# Ordenar días
tasa_cruce_canal["day_of_week"] = pd.Categorical(
    tasa_cruce_canal["day_of_week"], categories=dias, ordered=True
)


In [46]:
#heatmap

# Un heatmap por canal
canales = tasa_cruce_canal["contact"].unique()

fig = make_subplots(
    rows=1, cols=len(canales),
    subplot_titles=["cellular", "telephone", "unknown"],
)

for i, canal in enumerate(["cellular", "telephone", "unknown"], start=1):
    df_canal = (
        tasa_cruce_canal[tasa_cruce_canal["contact"] == canal]
        .pivot(index="day_of_week", columns="job_group", values="tasa_%")
         )
    text_values = df_canal.values.round(0).astype(object)
    
    text_values = [ 
        [f"{v:.0f}%" if not pd.isnull(v) else "" for v in row] # añadir % y Celdas NaN -> texto vacío
        for row in df_canal.values
        ] 

    

    fig.add_trace(
        go.Heatmap(
            z=df_canal.values,
            x=df_canal.columns.tolist(),
            y=df_canal.index.tolist(),
            colorscale="viridis",
            zmin=0, zmax=100,
            text= text_values, #df_canal.values.round(0),
            texttemplate="%{text}",
            showscale=False,
            colorbar=dict(title="Tasa (%)") if i == 3 else None
        ),
        row=1, col=i, 
    )

fig.update_layout(
    title="Heatmap: Tasa de Conversión por Día, Grupo de Trabajo y Canal",
    yaxis=dict(autorange="reversed"),
    yaxis2=dict(autorange="reversed"),
    yaxis3=dict(autorange="reversed"),
    )

fig.show()